# Q1 — Trial Landscape Overview

**Business Question:** What is the distribution of COVID-19 clinical trials by phase, status, and therapeutic area? How has this evolved over time?

---

In [2]:
# ── Setup ──────────────────────────────────────────────────────────────────
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine, text

# ─────────────────────────────────────────
# CONFIG — update username if needed
# ─────────────────────────────────────────
DB_USER = "vittoriobariosco"  # your username
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "clinical_trials" # your PostgreSQL database name 
CSV_PATH = "/Users/vittoriobariosco/Documents/APPLICATIONS/MIGx/data/processed/clinical_trials_clean.csv"  #path to cleaned CSV from EDA notebook

# ─────────────────────────────────────────
# 1. CONNECT TO POSTGRESQL
# ─────────────────────────────────────────
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")


# ── Global Plotly theme (minimal/corporate) ──────────────────────────────
COLORS = {
    'primary': '#2563EB',     # blue-600
    'secondary': '#64748B',   # slate-500
    'accent': '#F59E0B',      # amber-500
    'success': '#10B981',     # emerald-500
    'danger': '#EF4444',      # red-500
    'light_bg': '#F8FAFC',    # slate-50
    'text': '#1E293B',        # slate-800
    'muted': '#94A3B8',       # slate-400
}

# Categorical palette (up to 10 categories)
PALETTE = ['#2563EB', '#10B981', '#F59E0B', '#EF4444', '#8B5CF6',
           '#EC4899', '#14B8A6', '#F97316', '#6366F1', '#64748B']

# Reusable layout settings
LAYOUT_DEFAULTS = dict(
    font=dict(family='Helvetica Neue, Arial, sans-serif', color=COLORS['text']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=60, r=30, t=60, b=60),
    title_font_size=16,
    title_x=0.0,
    hoverlabel=dict(bgcolor='white', font_size=12),
)

def clean_axes(fig, show_grid_y=True):
    """Apply minimal axis styling."""
    fig.update_xaxes(showgrid=False, linecolor='#E2E8F0', linewidth=1)
    fig.update_yaxes(
        showgrid=show_grid_y,
        gridcolor='#F1F5F9',
        gridwidth=1,
        linecolor='#E2E8F0',
        linewidth=1
    )
    return fig

print('✅ Setup complete')

✅ Setup complete


---
## 1A — Distribution by Phase

In [3]:
# ── Query 1A ──────────────────────────────────────────────────────────────
query_1a = """
WITH phase_counts AS (
    SELECT phase, COUNT(*) AS trial_count
    FROM studies
    GROUP BY phase
)
SELECT
    phase,
    trial_count,
    ROUND(100.0 * trial_count / SUM(trial_count) OVER(), 1) AS pct
FROM phase_counts
ORDER BY trial_count DESC;
"""

df_phase = pd.read_sql(query_1a, engine)
df_phase

,phase,trial_count,pct
0,Unknown,2427,42.2
1,Not Applicable,1354,23.6
2,Phase 2,877,15.3
3,Phase 3,650,11.3
4,Phase 1,234,4.1
5,Phase 4,161,2.8
6,Early Phase 1,46,0.8


In [4]:
# ── Chart 1A: Horizontal bar chart — Phase distribution ─────────────────
# Reverse order so largest is at top
df_plot = df_phase.sort_values('trial_count', ascending=True)

fig_1a = go.Figure()

fig_1a.add_trace(go.Bar(
    x=df_plot['trial_count'],
    y=df_plot['phase'],
    orientation='h',
    marker_color=[COLORS['secondary'] if p == 'Unknown' else COLORS['primary']
                  for p in df_plot['phase']],
    text=[f"{v:,}  ({p}%)" for v, p in zip(df_plot['trial_count'], df_plot['pct'])],
    textposition='outside',
    textfont=dict(size=11),
))

fig_1a.update_layout(
    **LAYOUT_DEFAULTS,
    title='Distribution of Trials by Phase',
    xaxis_title='Number of Trials',
    yaxis_title='',
    height=400,
    width=750,
    showlegend=False,
)
clean_axes(fig_1a)
fig_1a.show()

### Findings — Phase Distribution

**1. Massive data gap:** 42.2% of trials (2,427) have no declared phase. Any phase-specific analysis must acknowledge this gap and exclude "Unknown" explicitly.

**2. "Not Applicable" (23.6%).** These are primarily non-drug interventions: behavioral studies, device trials, diagnostic evaluations. Combined with Unknown, **65.8% of COVID-19 trials have no meaningful phase label**.

**3. Among drug development phases:**
- Phase 2 leads (877 trials, 15.3%).
- Phase 3 follows (650, 11.3%).
- Phase 1 is relatively small (234, 4.1%).
- Phase 4 (161, 2.8%).
- Early Phase 1 (46, 0.8%).

---

## 1B — Distribution by Status

In [5]:
# ── Query 1B ──────────────────────────────────────────────────────────────
query_1b = """
WITH group_level AS (
    SELECT status_group, COUNT(*) AS group_count
    FROM studies
    GROUP BY status_group
),
detail_level AS (
    SELECT status_group, status, COUNT(*) AS status_count
    FROM studies
    GROUP BY status_group, status
)
SELECT
    d.status_group,
    d.status,
    d.status_count,
    g.group_count,
    ROUND(100.0 * d.status_count / g.group_count, 1) AS pct_within_group,
    ROUND(100.0 * d.status_count / SUM(d.status_count) OVER(), 1) AS pct_overall
FROM detail_level d
JOIN group_level g ON d.status_group = g.status_group
ORDER BY g.group_count DESC, d.status_count DESC;
"""

df_status = pd.read_sql(query_1b, engine)
df_status

,status_group,status,status_count,group_count,pct_within_group,pct_overall
0,Active,Recruiting,2805,4516,62.1,48.8
1,Active,Not yet recruiting,1004,4516,22.2,17.5
2,Active,"Active, not recruiting",526,4516,11.6,9.1
3,Active,Enrolling by invitation,181,4516,4.0,3.1
4,Completed,Completed,1025,1025,100.0,17.8
5,Stopped,Withdrawn,107,208,51.4,1.9
6,Stopped,Terminated,74,208,35.6,1.3
7,Stopped,Suspended,27,208,13.0,0.5


In [6]:
# ── Chart 1B-1: Donut chart — Status group level ─────────────────────────
df_groups = df_status.groupby('status_group')['status_count'].sum().reset_index()
df_groups.columns = ['status_group', 'count']
df_groups = df_groups.sort_values('count', ascending=False)

status_colors = {
    'Active': COLORS['primary'],
    'Completed': COLORS['success'],
    'Stopped': COLORS['danger']
}

fig_1b1 = go.Figure(go.Pie(
    labels=df_groups['status_group'],
    values=df_groups['count'],
    hole=0.55,
    marker_colors=[status_colors[s] for s in df_groups['status_group']],
    textinfo='label+percent',
    textfont_size=12,
    hovertemplate='%{label}: %{value:,} trials (%{percent})<extra></extra>',
    pull=[0.02, 0, 0],
))

fig_1b1.update_layout(
    **LAYOUT_DEFAULTS,
    title='Trial Status Distribution',
    height=400,
    width=500,
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.5),
    annotations=[dict(
        text=f"<b>{df_groups['count'].sum():,}</b><br>trials",
        x=0.5, y=0.5, font_size=16, showarrow=False,
        font_color=COLORS['text']
    )]
)
fig_1b1.show()

In [7]:
# ── Chart 1B-2: Breakdown within each status group ───────────────────────
fig_1b2 = go.Figure()

for i, group in enumerate(['Active', 'Completed', 'Stopped']):
    subset = df_status[df_status['status_group'] == group].sort_values('status_count', ascending=True)
    fig_1b2.add_trace(go.Bar(
        x=subset['status_count'],
        y=subset['status'],
        orientation='h',
        name=group,
        marker_color=status_colors[group],
        text=[f"{v:,} ({p}%)" for v, p in zip(subset['status_count'], subset['pct_within_group'])],
        textposition='outside',
        textfont=dict(size=10),
    ))

fig_1b2.update_layout(
    **LAYOUT_DEFAULTS,
    title='Detailed Status Breakdown within Each Group',
    xaxis_title='Number of Trials',
    yaxis_title='',
    height=450,
    width=800,
    barmode='stack',
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=-0.2, xanchor='center', x=0.5),
)
clean_axes(fig_1b2)
fig_1b2.show()

### Key Findings — Status Distribution

**1. The vast majority of trials are still active (78.6%, 4,516 trials).** This is expected given the extraction date (April 2021) — most COVID trials were launched in 2020 and had not yet reached completion.

**2. Only 17.8% of trials (1,025) have completed.** Among the active group:
- **Recruiting** (62.1% of active, 2,805 trials) — still enrolling patients.
- **Not yet recruiting** (22.2%, 1,004) — approved but not started enrollment.
- **Active, not recruiting** (11.6%, 526) — enrolled, follow-up in progress.
- **Enrolling by invitation** (4.0%, 181) — limited access trials.

**3. The failure rate appears low (3.6%, 208 stopped trials).** Many Active trials will eventually fail — the true failure rate can only be assessed once all trials resolve. Among the stopped:
- **Withdrawn** (51.4%, 107) — canceled before enrollment. Signals design or feasibility issues.
- **Terminated** (35.6%, 74) — stopped after partial enrollment. Signals safety, futility, or funding issues.
- **Suspended** (13.0%, 27) — temporarily stopped.

**4. Observations** Report the completion rate among *resolved* trials only (Completed + Stopped): **1,025 / 1,233 = 83.1%**. This is the meaningful metric.

---

## 1C — Distribution by Therapeutic Area

In [8]:
# ── Query 1C ──────────────────────────────────────────────────────────────
query_1c = """
WITH area_counts AS (
    SELECT
        c.therapeutic_area,
        COUNT(DISTINCT s.study_id) AS trial_count
    FROM studies s
    JOIN conditions c ON s.study_id = c.study_id
    GROUP BY c.therapeutic_area
),
total AS (
    SELECT COUNT(*) AS n FROM studies
)
SELECT
    a.therapeutic_area,
    a.trial_count,
    ROUND(100.0 * a.trial_count / t.n, 1) AS pct_of_all_studies
FROM area_counts a
CROSS JOIN total t
ORDER BY a.trial_count DESC;
"""

df_area = pd.read_sql(query_1c, engine)
df_area

,therapeutic_area,trial_count,pct_of_all_studies
0,Infectious Disease,4600,80.0
1,Other,1529,26.6
2,Respiratory/Critical Care,752,13.1
3,Mental Health,417,7.3
4,Metabolic/Cardiovascular,305,5.3
5,Oncology,165,2.9
6,Neurological,90,1.6
7,Musculoskeletal/Rheumatology,80,1.4
8,Renal,76,1.3
9,Reproductive Health,52,0.9


In [9]:
# ── Chart 1C: Horizontal bar — Therapeutic area ──────────────────────────
df_plot_c = df_area.sort_values('trial_count', ascending=True)

fig_1c = go.Figure()

fig_1c.add_trace(go.Bar(
    x=df_plot_c['trial_count'],
    y=df_plot_c['therapeutic_area'],
    orientation='h',
    marker_color=COLORS['primary'],
    text=[f"{v:,}  ({p}%)" for v, p in zip(df_plot_c['trial_count'], df_plot_c['pct_of_all_studies'])],
    textposition='outside',
    textfont=dict(size=11),
))

fig_1c.update_layout(
    **LAYOUT_DEFAULTS,
    title='Trials by Therapeutic Area (multi-label — percentages sum to >100%)',
    xaxis_title='Number of Distinct Trials',
    yaxis_title='',
    height=450,
    width=800,
    showlegend=False,
)
clean_axes(fig_1c)
fig_1c.show()

### Key Findings — Therapeutic Area Distribution

**Note:** One trial can target multiple conditions across different therapeutic areas. Percentages are computed against the total 5,749 studies and sum to >100%. This is standard for multi-label classification.

**1. Infectious Disease dominates at 80.0% (4,600 trials).** This is the direct COVID-19 response — trials targeting the virus itself, transmission, and infection management.

**2. "Other" captures 26.6% (1,529 trials).** This includes non-disease endpoints. This category is deliberately broad, further sub-classification was not attempted.

**3. Respiratory/Critical Care ranks third (13.1%, 752 trials).** COVID's primary clinical manifestation — pneumonia, ARDS, mechanical ventilation studies. Many trials are tagged both as Infectious Disease AND Respiratory (multi-label).

**4. Mental Health shows a notable 7.3% (417 trials)** 

**5. The long tail** (Oncology 2.9%, Neurological 1.6%, Musculoskeletal 1.4%, Renal 1.3%, Reproductive 0.9%).

---

## 1D — Temporal Evolution

In [11]:
# ── Query 1D ──────────────────────────────────────────────────────────────
query_1d = """
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', start_date) AS month,
        COUNT(*) AS total_started,
        COUNT(*) FILTER (WHERE status_group = 'Active')    AS active,
        COUNT(*) FILTER (WHERE status_group = 'Completed') AS completed,
        COUNT(*) FILTER (WHERE status_group = 'Stopped')   AS stopped
    FROM studies
    WHERE
        pre_covid_start = FALSE
        AND start_date IS NOT NULL
    GROUP BY DATE_TRUNC('month', start_date)
)
SELECT
    month,
    total_started,
    active,
    completed,
    stopped,
    SUM(total_started) OVER(ORDER BY month) AS cumulative_total
FROM monthly
ORDER BY month;
"""

df_timeline = pd.read_sql(query_1d, engine)
df_timeline['month'] = pd.to_datetime(df_timeline['month'], utc=True)
df_timeline

,month,total_started,active,completed,stopped,cumulative_total
0,2019-12-31 23:00:00+00:00,61,42,17,2,61.0
1,2020-01-31 23:00:00+00:00,100,57,32,11,161.0
2,2020-02-29 23:00:00+00:00,417,233,175,9,578.0
3,2020-03-31 22:00:00+00:00,825,539,226,60,1403.0
4,2020-04-30 22:00:00+00:00,645,440,178,27,2048.0
5,2020-05-31 22:00:00+00:00,502,373,119,10,2550.0
6,2020-06-30 22:00:00+00:00,361,274,73,14,2911.0
7,2020-07-31 22:00:00+00:00,265,215,42,8,3176.0
8,2020-08-31 22:00:00+00:00,306,247,47,12,3482.0
9,2020-09-30 22:00:00+00:00,257,231,20,6,3739.0


In [12]:
# ── Chart 1D-1: Monthly trials started (stacked bar by status) ──────────
# Focus on the main period: Jan 2020 — Jun 2021
df_main = df_timeline[df_timeline['month'] <= '2021-06-01'].copy()
df_main['month_label'] = df_main['month'].dt.strftime('%b %Y')

fig_1d1 = go.Figure()

for col, name, color in [
    ('completed', 'Completed', COLORS['success']),
    ('active', 'Active', COLORS['primary']),
    ('stopped', 'Stopped', COLORS['danger']),
]:
    fig_1d1.add_trace(go.Bar(
        x=df_main['month_label'],
        y=df_main[col],
        name=name,
        marker_color=color,
        hovertemplate=f'{name}: %{{y}}<extra></extra>',
    ))

fig_1d1.update_layout(
    **LAYOUT_DEFAULTS,
    title='Monthly Trials Started by Current Status (Jan 2020 – Jun 2021)',
    xaxis_title='',
    yaxis_title='Trials Started',
    barmode='stack',
    height=450,
    width=900,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    xaxis_tickangle=-45,
)
clean_axes(fig_1d1)
fig_1d1.show()

In [13]:
# ── Chart 1D-2: Cumulative trials over time (line chart) ─────────────────
fig_1d2 = go.Figure()

fig_1d2.add_trace(go.Scatter(
    x=df_main['month'],
    y=df_main['cumulative_total'],
    mode='lines+markers',
    line=dict(color=COLORS['primary'], width=2.5),
    marker=dict(size=5, color=COLORS['primary']),
    fill='tozeroy',
    fillcolor='rgba(37, 99, 235, 0.08)',
    hovertemplate='%{x|%b %Y}: %{y:,} trials<extra></extra>',
    name='Cumulative Trials',
))

# Annotate key milestones
fig_1d2.add_annotation(
    x='2020-04-01', y=1403,
    text='Peak month: 825 trials<br>started in Apr 2020',
    showarrow=True, arrowhead=2, arrowcolor=COLORS['secondary'],
    font=dict(size=10, color=COLORS['text']),
    bgcolor='white', bordercolor=COLORS['secondary'], borderwidth=1,
    ax=60, ay=-40,
)

fig_1d2.add_annotation(
    x='2020-06-01', y=2550,
    text='50% of all trials launched<br>by Jun 2020 (6 months)',
    showarrow=True, arrowhead=2, arrowcolor=COLORS['secondary'],
    font=dict(size=10, color=COLORS['text']),
    bgcolor='white', bordercolor=COLORS['secondary'], borderwidth=1,
    ax=70, ay=-30,
)

fig_1d2.update_layout(
    **LAYOUT_DEFAULTS,
    title='Cumulative COVID-19 Trials Over Time',
    xaxis_title='',
    yaxis_title='Cumulative Trials',
    height=420,
    width=900,
    showlegend=False,
)
clean_axes(fig_1d2)
fig_1d2.show()

### Key Findings — Temporal Evolution

**1. The global response.** Trial launches went from 61 in January 2020 to a peak of **825 in April 2020** — a 13.5× increase in just 3 months. This reflects the massive mobilization of academic, government, and industry resources.

**2. The first half of 2020 concentrates the majority of launches.** By June 2020, just 6 months into the pandemic, over 2,500 trials (approximately 50% of the total) had already been registered. The cumulative curve shows a classic S-shape: rapid acceleration (Mar–Jun 2020), gradual deceleration (Jul–Dec 2020), and plateau (2021).

**3. Stopped trials concentrate in the first surge (Mar–May 2020).** April 2020 alone accounts for 60 of the 208 stopped trials (28.8%).

**4. Completed trials are predominantly from early-2020 launches.** Trials started in Feb–May 2020 show the highest completion counts — they had 12+ months to reach endpoints. Later launches show fewer completions simply because they had less time by the extraction date (April 2021).

**5. 2021 shows a sharp decline in new registrations** (125 in April 2021, down to single digits by mid-year). This signals the transition from the "discovery" phase to the "consolidation" phase of pandemic research.

---

## Summary & Recommendations


1. **Phase data is unreliable for portfolio-level conclusions.** With 42.2% missing and 23.6% "Not Applicable", only ~34% of trials have a traditional phase label. Clients should not over-index on phase distribution. Recommendation: analyze Interventional trials with known phases separately (n ≈ 1,968) for phase-specific insights.

2. **Data freshness warning:** The dataset was extracted in April 2021. All status-based metrics are snapshots. A follow-up extraction (2022 or later) would reveal the true completion and failure rates.